# Chapter 5: Calculating with Points on Lines

**Source span.** Perspectives on Projective Geometry, Chapter 5, Sections 5.1-5.7, printed pages 79-92 / PDF pages 101-114.

**Chapter goal.** Turn points on a projective line into arithmetic without smuggling in a metric ruler. The working tools are harmonic quadruples, a projective scale with anchors `0`, `1`, and `infinity`, and join/meet constructions that realize addition and multiplication.

**Chapter question.** How can incidence constructions on a line begin to act like the real number field?

This notebook is a standalone reconstruction of the chapter's mathematical route. It uses the source for structure and terminology only; the prose, code, checks, and figures are original. Every visual is generated from coordinates or exact algebra, then checked by cross-ratio, incidence, or arithmetic residuals.

In [ ]:
from pathlib import Path
import sys
START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-05-calculating-with-points-on-lines"
for child in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.relative_to(BOOK_ROOT)

## Computational Translation And Planner Pass

| Chapter idea | Computational object | What gets checked |
| --- | --- | --- |
| Point on `RP^1` | Homogeneous pair `(u, v)` | Determinant cross-ratio |
| Point on the working line in `RP^2` | Homogeneous triple `(x, 0, z)` | Incidence with the base line |
| Harmonic position | `(a,b;c,d) = -1` | Invariance after construction choices |
| Projective scale | Anchors `0`, `1`, `infinity` | `(0,infinity; x,1)` recovers `x` |
| Harmonic map fixing anchors | Induced function `f : R -> R` | Field operations are forced |
| Von Staudt arithmetic | Join/meet recipes | Coordinates `x+y` and `x*y` |
| Pappos arithmetic | Swapped multiplication incidence | Same endpoint for `x*y` and `y*x` |

Planner choices: `SymPy` checks the exact cross-ratio identities; `Matplotlib` renders durable 2D join/meet constructions; `Plotly` gives a hoverable projective-scale lab; `NetworkX` exposes proof dependencies. Mesh, topology, GIS, and image-geometry libraries are not used because the source span is about one-dimensional projective arithmetic and planar incidence diagrams.

In [ ]:
import math, textwrap
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from utils.artifacts import assert_artifacts, book_relative, display_artifact, save_json, save_table

plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 180, "font.size": 10})
TOL = 1e-9

def H(x, y, w=1.0): return np.array([float(x), float(y), float(w)], dtype=float)
def nh(p, tol=1e-12):
    p=np.asarray(p,dtype=float); i=int(np.argmax(np.abs(p)))
    return p.copy() if abs(p[i]) < tol else p/p[i]
def join(p,q): return np.cross(np.asarray(p,dtype=float), np.asarray(q,dtype=float))
def meet(l,m): return np.cross(np.asarray(l,dtype=float), np.asarray(m,dtype=float))
def aff(p):
    p=np.asarray(p,dtype=float)
    if abs(p[2]) < 1e-12: raise ValueError("point at infinity")
    return p[:2]/p[2]
def base(t): return H(t,0,1)
def lp(t): return np.array([1.0,0.0]) if t is math.inf or t == math.inf else np.array([float(t),1.0])
def det2(a,b): return float(a[0]*b[1]-a[1]*b[0])
def cr(a,b,c,d): return det2(a,c)*det2(b,d)/(det2(a,d)*det2(b,c))
def pgl2(M,p): return np.asarray(M,dtype=float) @ np.asarray(p,dtype=float)
def coord(p): return math.inf if abs(p[1]) < 1e-12 else float(p[0]/p[1])
def xcoord(P): return float(aff(nh(P))[0])
def draw(ax, pts, **kw):
    xy=np.array([aff(p) for p in pts]); ax.plot(xy[:,0], xy[:,1], **kw)
def dot(ax, p, label, dx=.03, dy=.04, color="#111", size=36):
    xy=aff(p); ax.scatter([xy[0]],[xy[1]],s=size,color=color,zorder=5); ax.text(xy[0]+dx,xy[1]+dy,label,color=color,fontsize=9,zorder=6)
def savefig(fig, name):
    path=ARTIFACT_ROOT/"figures"/name; fig.savefig(path,bbox_inches="tight"); plt.close(fig); return path

ZERO, ONE, INF = base(0), base(1), H(1,0,0)
BASE = join(ZERO, INF)

STORYBOARD = {
 "chapter_goal":"Recover arithmetic on a projective line from harmonic quadruples, anchors, and incidence constructions.",
 "source_span_read":{"printed_pages":"79-92","pdf_pages":"101-114","sections":"5.1-5.7"},
 "concept_inventory":["harmonic position","projective scales","real-number reconstruction","von Staudt addition","von Staudt multiplication","Pappos arithmetic"],
 "library_routing_table":[
  {"concept":"cross-ratio identities","library":"SymPy","why":"exact determinant identities"},
  {"concept":"join/meet constructions","library":"Matplotlib","why":"planar incidence diagrams with durable labels"},
  {"concept":"anchor scale lab","library":"Plotly","why":"hover inspection of source and distorted coordinates"},
  {"concept":"proof dependencies","library":"NetworkX","why":"directed theorem dependencies"}],
 "visual_sequence":[
  {"artifact":"figures/harmonic-fourth-point-construction.png","validation":"cross-ratio -1 and auxiliary independence"},
  {"artifact":"figures/projective-scale-anchors-reconstruction.png","validation":"anchor reconstruction residual"},
  {"artifact":"html/projective-scale-anchor-lab.html","validation":"same reconstruction table"},
  {"artifact":"figures/field-automorphism-dependency-diagram.png","validation":"symbolic harmonic identities"},
  {"artifact":"figures/von-staudt-addition-construction.png","validation":"coordinate x+y"},
  {"artifact":"figures/von-staudt-multiplication-construction.png","validation":"coordinate x*y"},
  {"artifact":"figures/pappos-multiplication-commutativity.png","validation":"product and Pappus residuals"}],
 "artifact_plan":{"figures":6,"html":1,"tables":4,"checks":["storyboard.json","visual-checks.json","final-sanity.json"]},
 "computational_checks":["cross-ratio","symbolic identities","addition/multiplication residuals","Pappos residuals","artifact existence"],
 "proof_visualization_strategy":"dependency graph plus Pappos/von Staudt overlay",
 "risks":["finite drawings can hide the point at infinity","ill-conditioned auxiliary choices"],
 "acceptance_criteria":["Chapter 5 nbclient execution passes","scoped audits pass","scoped git diff --check passes"]}
storyboard_path = save_json(STORYBOARD, ARTIFACT_ROOT, "checks", "storyboard.json")
pd.DataFrame(STORYBOARD["visual_sequence"])

## Harmonic Points: The Fourth Point Is Forced

A harmonic quadruple is a pair of point-pairs `(a,b)` and `(c,d)` on one projective line with cross-ratio `(a,b;c,d) = -1`. The value `-1` is special because swapping either pair leaves it unchanged. Once `a`, `b`, and `c` are fixed, there is only one possible fourth point `d`.

The construction below uses an auxiliary point `o` off the base line and a second auxiliary point `p` on the line through `o` and `c`. Two different auxiliary choices are drawn. The inspection target is `d`: the construction web moves, but the fourth harmonic point does not.

In [ ]:
x,y=sp.symbols("x y", nonzero=True)
def lps(v): return sp.Matrix([1,0]) if v=="inf" else sp.Matrix([v,1])
def d2(a,b): return sp.Matrix([[a[0],b[0]],[a[1],b[1]]]).det()
def crs(a,b,c,d): return sp.simplify(d2(a,c)*d2(b,d)/(d2(a,d)*d2(b,c)))
ids=[
 ("(-x,x;0,infinity)",crs(lps(-x),lps(x),lps(0),lps("inf"))),
 ("(0,2x;x,infinity)",crs(lps(0),lps(2*x),lps(x),lps("inf"))),
 ("(x,y;(x+y)/2,infinity)",crs(lps(x),lps(y),lps((x+y)/2),lps("inf"))),
 ("(-1,1;x,1/x)",crs(lps(-1),lps(1),lps(x),lps(1/x))),
 ("(-x,x;1,x^2)",crs(lps(-x),lps(x),lps(1),lps(x**2)))]
harmonic_rows=[]
for name,val in ids:
    assert sp.simplify(val+1)==0
    harmonic_rows.append({"identity":name,"computed_cross_ratio":str(val),"residual_from_minus_one":str(sp.simplify(val+1)),"chapter_use":"harmonic preservation forces one arithmetic rule in the projective scale","inspection_target":"exact simplification must leave zero residual"})
harmonic_table=save_table(harmonic_rows,ARTIFACT_ROOT,"tables","harmonic-identities.csv")

def blend(p,q,t): return (1-t)*np.asarray(p,dtype=float)+t*np.asarray(q,dtype=float)
def harmonic_fourth(a,b,c,o,paux):
    A,B,C=base(a),base(b),base(c); ell=join(A,B)
    u=meet(join(o,A),join(paux,B)); v=meet(join(o,B),join(paux,A)); D=nh(meet(ell,join(u,v)))
    return {"A":A,"B":B,"C":C,"O":o,"P":paux,"U":u,"V":v,"D":D}
A0,B0,C0=0.0,1.0,0.35
O1=H(-.55,1.35,1); P1=blend(O1,base(C0),.42)
O2=H(1.55,1.15,1); P2=blend(O2,base(C0),.57)
C1=harmonic_fourth(A0,B0,C0,O1,P1); C2=harmonic_fourth(A0,B0,C0,O2,P2)
D1,D2=xcoord(C1["D"]),xcoord(C2["D"])
harmonic_cr_error=abs(cr(lp(A0),lp(B0),lp(C0),lp(D1))+1)
harmonic_auxiliary_error=abs(D1-D2)
fig,ax=plt.subplots(figsize=(9,4.6)); ax.axhline(0,color="#222",lw=2); ax.set_xlim(-1.55,1.75); ax.set_ylim(-.2,1.55); ax.set_yticks([])
ax.set_title("Fourth harmonic point from two auxiliary choices"); ax.set_xlabel("affine chart on the base line")
for pack,color,label in [(C1,"#2563eb","choice 1"),(C2,"#0891b2","choice 2")]:
    for a,b in [("O","A"),("O","B"),("O","C"),("P","A"),("P","B"),("U","V")]: draw(ax,[pack[a],pack[b]],color=color,lw=1.2,alpha=.8)
    draw(ax,[pack["O"],pack["P"]],color=color,lw=1,ls="--",alpha=.45); dot(ax,pack["O"],"o",color=color,size=28); dot(ax,pack["P"],"p",color=color,size=28)
    ax.plot([],[],color=color,label=label)
for val,lab in [(A0,"a=0"),(C0,"c"),(B0,"b=1"),(D1,"d")]: dot(ax,base(val),lab,dy=-.12,size=38)
ax.legend(frameon=False,loc="upper left"); ax.text(-1.48,1.33,f"(a,b;c,d) = {cr(lp(A0),lp(B0),lp(C0),lp(D1)):.12g}"); ax.text(-1.48,1.20,f"two auxiliary choices differ by {harmonic_auxiliary_error:.2e}")
harmonic_figure=savefig(fig,"harmonic-fourth-point-construction.png")
display_artifact(harmonic_figure,width=760)
{"fourth_harmonic_coordinate":float(D1),"cross_ratio_error":float(harmonic_cr_error),"auxiliary_independence_error":float(harmonic_auxiliary_error)}

## Projective Scales With Anchors

A projective scale is the choice of three distinct anchors on a line, named `0`, `1`, and `infinity`. After those anchors are fixed, the coordinate of a point is reconstructed by `(0,infinity; x,1)`. The visible spacing can change under a projective reparametrization; the anchor-based coordinate does not.

In [ ]:
values=np.array([-1.40,-.55,0,.40,1,1.85,2.60],dtype=float)
M=np.array([[1.20,-.35],[.25,.90]])
a0,a1,ai=pgl2(M,lp(0)),pgl2(M,lp(1)),pgl2(M,lp(math.inf))
rows=[]; src=[]; img=[]; rec=[]
for v in values:
    q=lp(float(v)); mq=pgl2(M,q); r=cr(a0,ai,mq,a1)
    src.append(coord(q)); img.append(coord(mq)); rec.append(r)
    rows.append({"source_coordinate":float(v),"distorted_chart_coordinate":float(coord(mq)),"recovered_anchor_coordinate":float(r),"absolute_error":float(abs(r-v))})
scale_reconstruction_error=max(r["absolute_error"] for r in rows)
scale_table=save_table(rows,ARTIFACT_ROOT,"tables","projective-scale-reconstruction.csv")
fig,axes=plt.subplots(2,1,figsize=(9,4.8),sharex=False)
for ax,coords,title in [(axes[0],src,"source chart"),(axes[1],img,"distorted chart after PGL(2,R)")]:
    ax.axhline(0,color="#1f2937",lw=2); ax.set_yticks([]); ax.set_ylim(-.3,.45); ax.set_title(title,loc="left")
    for c,v in zip(coords,values):
        color="#dc2626" if v in [0,1] else "#2563eb"; ax.scatter([c],[0],color=color,s=45,zorder=3); ax.text(c,.12,f"{v:g}",ha="center",fontsize=9)
axes[1].scatter([coord(ai)],[0],color="#7c3aed",s=55); axes[1].text(coord(ai),-.18,"image of infinity",ha="center",fontsize=9,color="#7c3aed"); axes[1].set_xlabel("displayed chart coordinate")
fig.suptitle("Projective scale: anchors move, recovered coordinates do not",y=.98)
scale_figure=savefig(fig,"projective-scale-anchors-reconstruction.png")
pl=go.Figure()
pl.add_trace(go.Scatter(x=src,y=[1]*len(src),mode="markers+text",text=[f"{v:g}" for v in values],textposition="top center",marker=dict(size=10,color="#2563eb"),name="source",hovertemplate="source %{text}<extra></extra>"))
pl.add_trace(go.Scatter(x=img,y=[0]*len(img),mode="markers+text",text=[f"{v:g}" for v in values],customdata=np.column_stack([rec,img]),textposition="bottom center",marker=dict(size=10,color="#059669"),name="distorted",hovertemplate="display %{customdata[1]:.4f}<br>recovered %{customdata[0]:.4f}<extra></extra>"))
pl.add_trace(go.Scatter(x=[coord(a0),coord(a1),coord(ai)],y=[-.18,-.18,-.18],mode="markers+text",text=["0 anchor","1 anchor","infinity anchor"],textposition="bottom center",marker=dict(size=12,color="#7c3aed"),name="image anchors"))
pl.update_layout(title="Projective scale anchor lab",xaxis_title="visible chart coordinate",yaxis=dict(visible=False),height=520,template="plotly_white")
scale_html=ARTIFACT_ROOT/"html"/"projective-scale-anchor-lab.html"; pl.write_html(scale_html,include_plotlyjs=True,full_html=True)
display_artifact(scale_figure,width=760); display_artifact(scale_html,width="100%",height=430)
{"max_reconstruction_error":float(scale_reconstruction_error),"html":book_relative(scale_html)}

## Reconstructing Real Numbers From Harmonic Maps

A harmonic map is a bijection of `RP^1` that sends harmonic quadruples to harmonic quadruples. If it also fixes `0`, `1`, and `infinity`, the projective scale reads it as a function `f : R -> R`. Harmonic identities force `f` to preserve midpoints, doubling, addition, negatives, squares, and products. That makes `f` a field automorphism of `R`; order and dense rationals force that automorphism to be the identity.

In [ ]:
field_rows=[
 {"forced_fact":"f(0)=0 and f(1)=1","harmonic_source":"anchors fixed","consequence":"scale endpoints rigid"},
 {"forced_fact":"f((x+y)/2)=(f(x)+f(y))/2","harmonic_source":"(x,y;(x+y)/2,infinity)=-1","consequence":"midpoints preserved"},
 {"forced_fact":"f(2x)=2f(x)","harmonic_source":"(0,2x;x,infinity)=-1","consequence":"doubling preserved"},
 {"forced_fact":"f(x+y)=f(x)+f(y)","harmonic_source":"midpoints plus doubling","consequence":"addition preserved"},
 {"forced_fact":"f(-x)=-f(x)","harmonic_source":"additivity and f(0)=0","consequence":"signs preserved"},
 {"forced_fact":"f(x^2)=f(x)^2","harmonic_source":"(-x,x;1,x^2)=-1","consequence":"squares preserved"},
 {"forced_fact":"f(x*y)=f(x)*f(y)","harmonic_source":"square identity applied to (x+y)^2","consequence":"products preserved"},
 {"forced_fact":"f is the identity on R","harmonic_source":"real order and dense rationals","consequence":"harmonic map is projective"}]
field_law_table=save_table(field_rows,ARTIFACT_ROOT,"tables","field-law-implications.csv")
field_identity_residuals={r["identity"]:r["residual_from_minus_one"] for r in harmonic_rows}; assert all(v=="0" for v in field_identity_residuals.values())
G=nx.DiGraph(); layers={"Harmonic quadruples preserved":0,"Anchors fixed":0,"Induced f on R":1,"Midpoints":2,"Doubling":2,"Addition":3,"Negatives":3,"Squares":3,"Products":4,"Field automorphism":5,"Identity over R":6,"Harmonic map is projective":7,"RP2 collineation is projective":8}
for n,l in layers.items(): G.add_node(n,layer=l)
G.add_edges_from([("Harmonic quadruples preserved","Induced f on R"),("Anchors fixed","Induced f on R"),("Induced f on R","Midpoints"),("Induced f on R","Doubling"),("Midpoints","Addition"),("Doubling","Addition"),("Addition","Negatives"),("Negatives","Squares"),("Squares","Products"),("Addition","Products"),("Addition","Field automorphism"),("Products","Field automorphism"),("Field automorphism","Identity over R"),("Identity over R","Harmonic map is projective"),("Harmonic map is projective","RP2 collineation is projective")])
pos={}; by={}
for n,l in layers.items(): by.setdefault(l,[]).append(n)
for l,ns in by.items():
    for i,n in enumerate(ns): pos[n]=(l,-i+(len(ns)-1)/2)
fig,ax=plt.subplots(figsize=(12,5.6)); colors=["#dbeafe" if layers[n]<5 else "#dcfce7" for n in G.nodes]
nx.draw_networkx_edges(G,pos,ax=ax,arrows=True,arrowstyle="-|>",arrowsize=14,width=1.2,edge_color="#64748b"); nx.draw_networkx_nodes(G,pos,ax=ax,node_size=2200,node_color=colors,edgecolors="#334155")
nx.draw_networkx_labels(G,pos,{n:textwrap.fill(n,16) for n in G.nodes},ax=ax,font_size=8); ax.set_title("Dependency diagram: harmonic preservation forces arithmetic"); ax.axis("off")
field_dependency_figure=savefig(fig,"field-automorphism-dependency-diagram.png")
display_artifact(field_dependency_figure,width=920)
{"nodes":G.number_of_nodes(),"edges":G.number_of_edges(),"symbolic_residuals":field_identity_residuals}

## Von Staudt Addition And Multiplication

The harmonic-map proof shows that arithmetic is forced, but it is not economical as a construction recipe. Von Staudt's gadgets build `x + y` and `x*y` directly from joins and meets. The affine chart below puts `infinity` in the horizontal direction, so lines aimed at `infinity` appear as horizontal parallels. The formulas themselves still use homogeneous joins and meets.

In [ ]:
P=H(0,1.55,1); Q=H(0,.58,1); TOP=join(P,INF); QH=join(Q,INF); P0=join(ZERO,P)
def vs_add(x,y):
    X,Y=base(x),base(y); q1=nh(meet(join(Q,INF),join(P,X))); r=nh(meet(join(P,INF),join(Q,Y))); z=nh(meet(BASE,join(r,q1)))
    return {"0":ZERO,"x":X,"y":Y,"z":z,"p":P,"q":Q,"q1":q1,"r":r}
def vs_mul(x,y):
    X,Y=base(x),base(y); r=nh(meet(join(P,INF),join(Q,X))); s=nh(meet(join(P,INF),join(Q,ONE))); q1=nh(meet(join(ZERO,P),join(s,Y))); z=nh(meet(BASE,join(r,q1)))
    return {"0":ZERO,"1":ONE,"x":X,"y":Y,"z":z,"p":P,"q":Q,"q1":q1,"r":r,"s":s}
def setup(ax,xmax,title):
    ax.axhline(0,color="#111827",lw=2.2); ax.axhline(aff(P)[1],color="#93c5fd",lw=1.1); ax.axhline(aff(Q)[1],color="#93c5fd",lw=1.1); draw(ax,[ZERO,P],color="#111827",lw=1.4,ls="--")
    ax.set_xlim(-1.15,xmax); ax.set_ylim(-.18,1.75); ax.set_aspect("equal",adjustable="box"); ax.set_yticks([]); ax.set_title(title,loc="left")
def plot_add(pack,x,y):
    fig,ax=plt.subplots(figsize=(9,4.5)); setup(ax,x+y+.9,"Von Staudt addition: transfer 0-to-x to start at y")
    draw(ax,[pack["p"],pack["x"]],color="#2563eb",lw=1.4); draw(ax,[pack["q"],pack["y"],pack["r"]],color="#2563eb",lw=1.4); draw(ax,[pack["r"],pack["q1"],pack["z"]],color="#2563eb",lw=1.4)
    for k,l in [("0","0"),("x","x"),("y","y"),("z","x+y")]: dot(ax,pack[k],l,dy=-.12)
    for k,l in [("p","p"),("q","q"),("q1","q1"),("r","r")]: dot(ax,pack[k],l,color="#1d4ed8",dx=-.16 if k=="q" else .03,dy=0 if k=="q" else .04)
    ax.text(-1.08,1.55,f"constructed coordinate = {xcoord(pack['z']):.6g}"); return savefig(fig,"von-staudt-addition-construction.png")
def plot_mul(pack,x,y):
    fig,ax=plt.subplots(figsize=(9,4.5)); setup(ax,x*y+.9,"Von Staudt multiplication: compare ratios 1:x and y:x*y")
    for pts in [["q","x","r"],["q","1","s"],["s","y","q1"],["r","q1","z"]]: draw(ax,[pack[i] for i in pts],color="#2563eb",lw=1.4)
    for k,l in [("0","0"),("1","1"),("x","x"),("y","y"),("z","x*y")]: dot(ax,pack[k],l,dy=-.12)
    for k,l in [("p","p"),("q","q"),("q1","q1"),("r","r"),("s","s")]: dot(ax,pack[k],l,color="#1d4ed8",dx=-.16 if k=="q" else .03,dy=0 if k=="q" else .04)
    ax.text(-1.08,1.55,f"constructed coordinate = {xcoord(pack['z']):.6g}"); return savefig(fig,"von-staudt-multiplication-construction.png")
XVAL,YVAL=1.45,2.20; add_pack=vs_add(XVAL,YVAL); mul_pack=vs_mul(XVAL,YVAL)
addition_error=abs(xcoord(add_pack["z"])-(XVAL+YVAL)); multiplication_error=abs(xcoord(mul_pack["z"])-(XVAL*YVAL))
h,k,xs,ys=sp.symbols("h k x y", nonzero=True); q1x=(h-k)*xs/h; rx=-(h-k)*ys/k; zadd=sp.simplify(rx+h/(h-k)*(q1x-rx)); A=(h-k)/k; rxm=-A*xs; q1y=h*ys/(ys+A); zmul=sp.simplify(rxm+h/(h-q1y)*(0-rxm))
assert sp.simplify(zadd-(xs+ys))==0 and sp.simplify(zmul-xs*ys)==0
addition_figure=plot_add(add_pack,XVAL,YVAL); multiplication_figure=plot_mul(mul_pack,XVAL,YVAL)
arithmetic_table=save_table([{"construction":"addition","input_x":XVAL,"input_y":YVAL,"expected":XVAL+YVAL,"constructed":xcoord(add_pack["z"]),"absolute_error":addition_error,"symbolic_formula":str(zadd),"inspection_target":"join/meet endpoint should sit at x plus y on the base line"},{"construction":"multiplication","input_x":XVAL,"input_y":YVAL,"expected":XVAL*YVAL,"constructed":xcoord(mul_pack["z"]),"absolute_error":multiplication_error,"symbolic_formula":str(zmul),"inspection_target":"join/meet endpoint should sit at x times y on the base line"}],ARTIFACT_ROOT,"tables","von-staudt-arithmetic-checks.csv")
display_artifact(addition_figure,width=760); display_artifact(multiplication_figure,width=760)
{"addition_error":float(addition_error),"multiplication_error":float(multiplication_error)}

## Pappos Arithmetic Checks

Addition is symmetric in the von Staudt construction: swapping `x` and `y` gives the same combinatorics. Multiplication is subtler. The two recipes for `x*y` and `y*x` overlay to form the incidence pattern governed by Pappos's theorem. Over the real field the two products coincide; in a purely synthetic setting, Pappos is the incidence axiom that forces this commutativity.

In [ ]:
def pappus_check():
    A,B,C=H(0,0,1),H(1.15,0,1),H(2.55,0,1); Ap,Bp,Cp=H(.25,1.10,1),H(1.35,1.10,1),H(2.85,1.10,1)
    X=nh(meet(join(A,Bp),join(Ap,B))); Y=nh(meet(join(A,Cp),join(Ap,C))); Z=nh(meet(join(B,Cp),join(Bp,C))); L=join(X,Y)
    return {"A":A,"B":B,"C":C,"Ap":Ap,"Bp":Bp,"Cp":Cp,"X":X,"Y":Y,"Z":Z,"line":L,"residual":abs(float(np.dot(Z,L)))/max(np.linalg.norm(L),TOL)}
mx,my=vs_mul(XVAL,YVAL),vs_mul(YVAL,XVAL); product_xy=xcoord(mx["z"]); product_yx=xcoord(my["z"]); pappos_product_residual=abs(product_xy-product_yx); pap=pappus_check()
fig,axes=plt.subplots(1,2,figsize=(12.8,4.9)); ax=axes[0]; setup(ax,XVAL*YVAL+.9,"Swapping x and y closes at one product point")
for pack,color,name in [(mx,"#2563eb","x*y"),(my,"#16a34a","y*x")]:
    for pts in [["q","x","r"],["q","1","s"],["s","y","q1"],["r","q1","z"]]: draw(ax,[pack[i] for i in pts],color=color,lw=1.2,alpha=.85,ls="-" if name=="x*y" else "--")
    ax.plot([],[],color=color,lw=1.7,label=name)
for k,l in [("0","0"),("1","1")]: dot(ax,mx[k],l,dy=-.12)
dot(ax,mx["x"],"x",dy=-.12,color="#2563eb"); dot(ax,mx["y"],"y",dy=-.12,color="#2563eb"); dot(ax,mx["z"],"x*y = y*x",dx=-.35,dy=-.12,size=46); dot(ax,P,"p"); dot(ax,Q,"q",dx=-.16,dy=0); ax.legend(frameon=False,loc="upper left"); ax.text(-1.05,1.52,f"product residual = {pappos_product_residual:.2e}")
ax=axes[1]; ax.set_title("Pappus collinearity residual",loc="left"); ax.axhline(0,color="#111827",lw=1.6); ax.axhline(1.10,color="#111827",lw=1.6)
for a,b in [("A","Bp"),("Ap","B"),("A","Cp"),("Ap","C"),("B","Cp"),("Bp","C")]: draw(ax,[pap[a],pap[b]],color="#93c5fd",lw=1)
draw(ax,[pap["X"],pap["Y"],pap["Z"]],color="#dc2626",lw=1.5)
for k,l in [("A","A"),("B","B"),("C","C"),("Ap","A'"),("Bp","B'"),("Cp","C'"),("X","P1"),("Y","P2"),("Z","P3")]: dot(ax,pap[k],l,color="#dc2626" if k in ["X","Y","Z"] else "#111",size=28)
ax.text(.05,-.42,f"collinearity residual = {pap['residual']:.2e}"); ax.set_xlim(-.35,3.15); ax.set_ylim(-.55,1.45); ax.set_aspect("equal",adjustable="box"); ax.set_yticks([])
pappos_figure=savefig(fig,"pappos-multiplication-commutativity.png")
display_artifact(pappos_figure,width=920)
{"product_xy":float(product_xy),"product_yx":float(product_yx),"product_residual":float(pappos_product_residual),"pappus_collinearity_residual":float(pap["residual"])}

## Artifact Gallery

The generated artifacts are linked here so the notebook remains readable before execution. The code cells above regenerate them from the same data.

![Harmonic fourth point construction](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/harmonic-fourth-point-construction.png)

![Projective scale anchors reconstruction](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/projective-scale-anchors-reconstruction.png)

![Field automorphism dependency diagram](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/field-automorphism-dependency-diagram.png)

![Von Staudt addition construction](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/von-staudt-addition-construction.png)

![Von Staudt multiplication construction](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/von-staudt-multiplication-construction.png)

![Pappos multiplication commutativity](../../artifacts/chapter-05-calculating-with-points-on-lines/figures/pappos-multiplication-commutativity.png)

Open the [projective scale anchor lab](../../artifacts/chapter-05-calculating-with-points-on-lines/html/projective-scale-anchor-lab.html) for hover inspection. The companion tables are `harmonic-identities.csv`, `projective-scale-reconstruction.csv`, `field-law-implications.csv`, and `von-staudt-arithmetic-checks.csv`.

## Applied Lab

1. Change `C0` in the harmonic construction. The auxiliary web should move, but the cross-ratio check should stay at `-1`.
2. Change the `2x2` matrix `M` in the scale section. The visible chart coordinates should change, but the recovered coordinates should remain the original sample values.
3. Change `XVAL` and `YVAL`. Von Staudt addition and multiplication should still return `x+y` and `x*y`, and the Pappos overlay should still close at one product point.
4. Move `Q` very close to `P`. The construction becomes numerically ill-conditioned even though the projective recipe is valid away from degeneracies.

## Takeaways

- Harmonic quadruples are the first nontrivial line invariant after collinearity becomes automatic.
- A projective scale turns three anchors into a coordinate recovery rule, but the rule is a cross-ratio, not a Euclidean measurement.
- A harmonic map fixing the anchors is forced to preserve the real field operations; over `R` this rigidity leaves only the identity after normalization.
- Von Staudt's addition and multiplication are efficient incidence constructions for arithmetic on a line.
- Pappos's theorem is the incidence statement hiding behind commutative multiplication in the synthetic setup.

In [ ]:
ARTIFACT_PATHS=[harmonic_figure,scale_figure,scale_html,field_dependency_figure,addition_figure,multiplication_figure,pappos_figure,harmonic_table,scale_table,field_law_table,arithmetic_table,storyboard_path]
assert_artifacts(ARTIFACT_PATHS)
visual_checks={
 "all_files_exist":all(p.exists() and p.stat().st_size>256 for p in ARTIFACT_PATHS),
 "cross_ratio_error":float(harmonic_cr_error),
 "harmonic_auxiliary_independence_error":float(harmonic_auxiliary_error),
 "scale_reconstruction_error":float(scale_reconstruction_error),
 "von_staudt_addition_error":float(addition_error),
 "von_staudt_multiplication_error":float(multiplication_error),
 "pappos_product_residual":float(pappos_product_residual),
 "pappus_collinearity_residual":float(pap["residual"]),
 "symbolic_harmonic_residuals":field_identity_residuals,
 "artifacts":[{"path":book_relative(p),"bytes":int(p.stat().st_size)} for p in ARTIFACT_PATHS]}
assert visual_checks["all_files_exist"]
for key in ["cross_ratio_error","harmonic_auxiliary_independence_error","scale_reconstruction_error","von_staudt_addition_error","von_staudt_multiplication_error","pappos_product_residual","pappus_collinearity_residual"]:
    assert visual_checks[key] < 1e-9, (key, visual_checks[key])
assert all(v=="0" for v in visual_checks["symbolic_harmonic_residuals"].values())
visual_checks_path=save_json(visual_checks,ARTIFACT_ROOT,"checks","visual-checks.json")
final_sanity={"chapter":5,"source_span":"printed pp. 79-92 / PDF pp. 101-114","notebook_executed":True,"checks_path":book_relative(visual_checks_path),"storyboard_path":book_relative(storyboard_path),"artifact_count":len(ARTIFACT_PATHS),"max_numeric_error":max(visual_checks[k] for k in ["cross_ratio_error","harmonic_auxiliary_independence_error","scale_reconstruction_error","von_staudt_addition_error","von_staudt_multiplication_error","pappos_product_residual","pappus_collinearity_residual"]),"displayed_artifacts":[book_relative(p) for p in ARTIFACT_PATHS if p.suffix.lower() in {".png",".html"}]}
final_sanity_path=save_json(final_sanity,ARTIFACT_ROOT,"checks","final-sanity.json")
final_sanity